In [1]:
import pandas as pd
import joblib
import time

from sklearn.model_selection import train_test_split
from skopt import BayesSearchCV  # Bayesian optimization: utilizado para optimizar hiperparámetros

import lightgbm as lgbm
from lightgbm import early_stopping  # Early stopping: utilizado para evitar sobreajuste

from Funcoes_Comuns import avaliar_modelo, registrar_modelo

### 1. Recuperar base já pré-processada

In [2]:
# Obter dados
df_enem = pd.read_pickle('Bases\\Finais\\enem_censo_2023_full.pkl')

In [3]:
#Variaveis alvo
variaveis_alvo = ['NUM_NOTA_MT', 'NUM_NOTA_LC', 'NUM_NOTA_CN', 'NUM_NOTA_CH', 'NUM_NOTA_REDACAO']
grupo_previsao = ['NUM_NOTA_CH']

# separar em treino e teste
X = df_enem.drop(columns=variaveis_alvo)
y = df_enem[grupo_previsao]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ajuste de tipo para MLflow -> Converter colunas inteiras para float
X_train = X_train.astype({col: 'float' for col in X_train.select_dtypes('int').columns})
X_test = X_test.astype({col: 'float' for col in X_test.select_dtypes('int').columns})

# Obter colunas categóricas
categorical_features = X_train.select_dtypes(include=['category']).columns.tolist()

# Criar Eval Set para validação cruzada (15% do conjunto de treino)
X_train_final, X_eval, y_train_final, y_eval = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42
)

In [4]:
# Ajustar as dimensões dos arrays
y_test = y_test.squeeze()
y_train_final = y_train_final.squeeze()
y_eval = y_eval.squeeze()

### 2. Modelo base

In [5]:
# Treinar modelo LGBMRegressor Base
modelo_lgbm = lgbm.LGBMRegressor(n_estimators=2000, 
                                 learning_rate=0.01, 
                                 max_depth=100,
                                 random_state=42,
                                 max_bin=4095,
                                 force_row_wise=True)

start_time = time.time()

modelo_lgbm.fit(X_train_final, 
                y_train_final, 
                eval_set=[(X_eval, y_eval)], 
                eval_metric=['r2', 'rmse', 'mae'],
                callbacks=[early_stopping(stopping_rounds=200)],
                categorical_feature=categorical_features)

tempo_treino = time.time() - start_time

[LightGBM] [Info] Total Bins 30747
[LightGBM] [Info] Number of data points in the train set: 487214, number of used features: 106
[LightGBM] [Info] Start training from score 527.881246
Training until validation scores don't improve for 200 rounds
[LightGBM] [Info] Start training from score 527.881246
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1295]	valid_0's rmse: 70.2518	valid_0's l1: 55.4426	valid_0's l2: 4935.32
Early stopping, best iteration is:
[1295]	valid_0's rmse: 70.2518	valid_0's l1: 55.4426	valid_0's l2: 4935.32


In [6]:
# Previsões
y_pred = modelo_lgbm.predict(X_test)

In [7]:
nome_experimento = 'Notas CH ENEM 2023'

registrar_modelo(experimento=nome_experimento,
                 parametros={**modelo_lgbm.get_params(), "amostra": X_train_final.shape[0], "tempo": tempo_treino},
                 X_train=X_train_final,
                 y_train=y_train_final,
                 y_test=y_test,
                 y_pred=y_pred,
                 variavel_alvo='NUM_NOTA_CH',
                 modelo=modelo_lgbm,
                 nome_modelo='modelo_lgbm_base_censo_enem',
                 descricao_modelo='Modelo LGBMRegressor base Censo e ENEM 2023')

2025/08/16 13:44:14 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
Successfully registered model 'modelo_lgbm_base_censo_enem'.
2025/08/16 13:44:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: modelo_lgbm_base_censo_enem, version 1
Successfully registered model 'modelo_lgbm_base_censo_enem'.
2025/08/16 13:44:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: modelo_lgbm_base_censo_enem, version 1


Modelo registrado com sucesso no MLflow: modelo_lgbm_base_censo_enem
🏃 View run abrasive-calf-311 at: http://127.0.0.1:9080/#/experiments/299918284299748162/runs/5d2708ddfaa04279b9fc6fb1f37ea654
🧪 View experiment at: http://127.0.0.1:9080/#/experiments/299918284299748162
Rastreamento do MLflow finalizado.


Created version '1' of model 'modelo_lgbm_base_censo_enem'.


In [8]:
# Avaliação grupo treino
avaliar_modelo(y_train_final, modelo_lgbm.predict(X_train_final), "treino")

# Avaliação grupo teste
avaliar_modelo(y_test, y_pred, "teste")

MAE (treino): 53.1164
RMSE (treino): 67.4018
R2 (treino): 0.3657
MAE (teste): 55.1669
RMSE (teste): 69.9683
R2 (teste): 0.3133


### 3. Bayes Search

In [5]:
modelo_lgbm_bayes = lgbm.LGBMRegressor(random_state=42,
                                       max_bin=4095, 
                                       force_row_wise=True)

In [ ]:
# Definição do espaço de busca otimizado para base unificada (111 colunas, 62 numéricas)
param_grid = {
    'num_leaves': (15, 75),                        # Número de folhas na árvore de decisão
    'max_depth': (40, 120),                        # Profundidade máxima da árvore
    'learning_rate': (0.005, 0.05, 'log-uniform'), # Taxa de aprendizado
    'n_estimators': (5200, 6500),                  # Número de árvores
    'subsample': (0.1, 0.9),                       # Proporção de amostras usadas em cada árvore
    'colsample_bytree': (0.1, 0.5),                # Fração de colunas a serem usadas por árvore
    'reg_alpha': (1e-4, 0.5, 'log-uniform'),       # Regularização L1
    'reg_lambda': (5e-7, 5e-4, 'log-uniform'),     # Regularização L2
}

In [11]:
# Configurar a busca Bayesiana otimizada para base unificada

# Criando o otimizador Bayesiano
bayes_search = BayesSearchCV(
    estimator=modelo_lgbm_bayes,    # Modelo a ser otimizado
    search_spaces=param_grid,       # Espaço de busca definido acima
    scoring='r2',                   # Critério de seleção
    n_iter=20,                      # Número de avaliações do modelo
    cv=3,                           # Validação cruzada
    random_state=42,                # Semente para reprodutibilidade
    n_jobs=-1,                      # Paralelização total dos cálculos
    verbose=1                       # 0 = sem mensagens, 1 = mensagens de progresso, 2 = mensagens detalhadas
)

In [12]:
fit_params = {
    'eval_metric': ['r2', 'rmse', 'mae'],                  # Métricas a serem avaliadas
    'categorical_feature': categorical_features,           # Colunas categóricas
}

In [13]:
# Executar a busca Bayesiana
start_time = time.time()
bayes_search.fit(X_train_final, y_train_final, **fit_params)

# Parar o cronômetro
end_time = time.time()
elapsed_time = end_time - start_time

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fi

In [ ]:
# Melhores parâmetros encontrados
try:
    melhores_parametros = bayes_search.best_params_
    print(f"Melhores parâmetros: {melhores_parametros}")
    print("R2: ", bayes_search.best_score_)
    print(f"Tempo total de execução: {elapsed_time:.2f} segundos")
except:
    melhores_parametros = {'colsample_bytree': 0.10216550758097204, 'learning_rate': 0.005101111879007455, 'max_depth': 69, 'n_estimators': 6086, 'num_leaves': 48, 'reg_alpha': 0.0002991765951731594, 'reg_lambda': 9.465059341080664e-05, 'subsample': 0.1049684128860605}
    print(f"Erro ao obter melhores parâmetros, usando valores calculados anteriormente:\n {melhores_parametros}")

Melhores parâmetros: OrderedDict([('colsample_bytree', 0.43880229485734434), ('feature_fraction', 0.5), ('learning_rate', 0.01), ('max_depth', 130), ('min_child_samples', 24), ('n_estimators', 4000), ('num_leaves', 50), ('reg_alpha', 8.967079369057016e-05), ('reg_lambda', 2.9956735374073784e-06), ('subsample', 0.95)])
R2:  0.3104601067863708
Tempo total de execução: 21941.39 segundos


In [7]:
# Treinar o modelo com os melhores parâmetros encontrados
modelo_lgbm_bayes.set_params(**melhores_parametros)

start_time = time.time()

# Treinamento do modelo com os melhores parâmetros encontrados
modelo_lgbm_bayes.fit(X_train_final, 
                      y_train_final, 
                      eval_set=[(X_eval, y_eval)], 
                      eval_metric=['r2', 'rmse', 'mae'],
                      callbacks=[early_stopping(stopping_rounds=200)],
                      categorical_feature=categorical_features)

tempo_treino = time.time() - start_time

[LightGBM] [Info] Total Bins 30747
[LightGBM] [Info] Number of data points in the train set: 487214, number of used features: 106
[LightGBM] [Info] Start training from score 527.881246
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[6058]	valid_0's rmse: 69.8465	valid_0's l1: 55.095	valid_0's l2: 4878.53


In [8]:
# Previsões
y_pred_bayes = modelo_lgbm_bayes.predict(X_test)

In [10]:
nome_experimento = 'Notas CH ENEM 2023'

registrar_modelo(experimento=nome_experimento,
                    modelo=modelo_lgbm_bayes,
                    parametros={**modelo_lgbm_bayes.get_params(), "amostra": X_train_final.shape[0], "tempo": tempo_treino},
                    X_train=X_train_final,
                    y_train=y_train_final,
                    y_test=y_test,
                    y_pred=y_pred_bayes,
                    variavel_alvo='NUM_NOTA_CH',
                    nome_modelo='modelo_lgbm_bayes_censo_enem',
                    descricao_modelo='Modelo LGBMRegressor otimizado com BayesSearchCV Censo e ENEM 2023')

2025/08/16 21:14:36 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
Registered model 'modelo_lgbm_bayes_censo_enem' already exists. Creating a new version of this model...
2025/08/16 21:15:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: modelo_lgbm_bayes_censo_enem, version 12


Modelo registrado com sucesso no MLflow: modelo_lgbm_bayes_censo_enem
🏃 View run kindly-bird-261 at: http://127.0.0.1:9080/#/experiments/957135083854196683/runs/850fdbb2c3bd4f5da4fcb2b13dbde236
🧪 View experiment at: http://127.0.0.1:9080/#/experiments/957135083854196683
Rastreamento do MLflow finalizado.


Created version '12' of model 'modelo_lgbm_bayes_censo_enem'.


In [9]:
# Avaliação grupo treino
avaliar_modelo(y_train_final, modelo_lgbm_bayes.predict(X_train_final), "treino")

# Avaliação grupo teste
avaliar_modelo(y_test, y_pred_bayes, "teste")

MAE (treino): 53.4328
RMSE (treino): 67.7899
R2 (treino): 0.3584
MAE (teste): 54.8751
RMSE (teste): 69.5892
R2 (teste): 0.3207


In [28]:
# Salvar modelo como Pickle
joblib.dump(modelo_lgbm_bayes, 'Modelos\\modelo_lgbm_bayes_censo_enem.pkl')

['Modelos\\modelo_lgbm_bayes_censo_enem.pkl']